# Lektion 08 - Multi-Agent-Entwurfsmuster


## Einrichtung


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Warum Multi-Agenten-Systeme?

Reale Aufgaben wie Reiseplanung erfordern viele verschiedene Fachkenntnisse — Logistik, Ortskenntnisse, Budgetierung und mehr. Ein einzelner Agent, der versucht, alles zu bewältigen, wird schnell unübersichtlich.

Multi-Agenten-Systeme lösen dies durch **Spezialisierung**: Jeder Agent konzentriert sich auf ein Fachgebiet und liefert dadurch hochwertigere Ergebnisse als ein Generalist. Sie verbessern auch die **Skalierbarkeit** — Sie können neue Agenten hinzufügen (z. B. einen Flugspezialisten, einen Restaurantkritiker), ohne den bestehenden Arbeitsablauf neu schreiben zu müssen. Die Agenten arbeiten zusammen in einer strukturierten Pipeline und geben den Kontext von einem zum nächsten weiter.


## Erstellung spezialisierter Agenten


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Erstellen eines sequenziellen Workflows

Mit `WorkflowBuilder` können Sie Agenten zu einem gerichteten Graphen verbinden. Hier erstellen wir eine einfache zweistufige Pipeline: Der **TravelPlanner** entwirft die Reiseroute, dann überprüft und verbessert der **TravelConcierge** sie.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Hinzufügen weiterer Agenten zum Workflow

Einer der größten Vorteile des Multi-Agenten-Musters ist, wie einfach es sich erweitern lässt. Unten fügen wir einen **BudgetReviewer**-Agenten hinzu, der den Plan mit dem Budget des Reisenden abgleicht, Elemente markiert, die die Kosten möglicherweise über das Limit hinaus treiben, und kostensparende Alternativen vorschlägt. Der Workflow führt jetzt drei Agenten nacheinander aus:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Zusammenfassung

In dieser Lektion haben Sie gelernt, wie man:

1. **Spezialisierte Agenten erstellt** — jeder mit einer fokussierten Rolle (Planung, Concierge, Budgetüberprüfung).
2. **Agenten in einen sequentiellen Arbeitsablauf integriert** mithilfe von `WorkflowBuilder` und `add_edge`.
3. **Ausgabe aus einer Multi-Agenten-Pipeline streamt**, wobei verfolgt wird, welcher Agent gerade spricht.
4. **Einen Arbeitsablauf erweitert**, indem neue Agenten zur Kette hinzugefügt werden, ohne bestehende zu verändern.

Das Multi-Agenten-Designmuster hält jeden Agenten einfach, während es reichhaltigere, gründlicher überprüfte Ergebnisse liefert, als es ein einzelner Agent alleine erreichen könnte.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:  
Dieses Dokument wurde mithilfe des KI-Übersetzungsdienstes [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir auf Genauigkeit achten, sollten Sie beachten, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in der ursprünglichen Sprache gilt als maßgebliche Quelle. Für wichtige Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die durch die Nutzung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
